## Measurement of shape and quality descriptors for parts of an object for a given dataset with pixel-level part annotations


### Import modules, set paths and load datasets

In [1]:
import os
import cv2
import json
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import scipy

from mahotas.features import zernike_moments
from brisque import BRISQUE # NR-IQA
from pypiqe import piqe # NR-IQA

from skimage import img_as_ubyte
from skimage.measure import regionprops, label, shannon_entropy
from skimage.transform import resize
from skimage.color import rgb2gray
from skimage.filters import laplace
from sklearn.model_selection import train_test_split
from skvideo.measure import niqe # NR-IQA

import torch
from torch import nn
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, Dataset
from torch.optim.lr_scheduler import ReduceLROnPlateau
import timm
import segmentation_models_pytorch as smp

# Patch imresize if missing
if not hasattr(scipy.misc, "imresize"):
    def imresize(arr, size, interp=None, mode=None):
        if isinstance(size, float):  # scale factor
            new_shape = (int(arr.shape[0] * size), int(arr.shape[1] * size))
        else:
            new_shape = size[:2]
        arr_resized = resize(arr, new_shape, order=3, mode="reflect", anti_aliasing=True)
        arr_resized = (arr_resized * 255).astype(np.uint8)
        return arr_resized
    scipy.misc.imresize = imresize

# Patch for deprecated NumPy aliases (for backward compatibility)
if not hasattr(np, 'int'):
    np.int = int
if not hasattr(np, 'float'):
    np.float = float
if not hasattr(np, 'bool'):
    np.bool = bool

Load a dataset with pixel-level part annotations for performing shape analysis

In [2]:
# Hyperparameters
DEVICE_ID = 0
BATCH_SIZE = 128
IMAGE_SIZE = 320

In [3]:
# Create SIFT and ORB detectors once
sift = cv2.SIFT_create()
orb = cv2.ORB_create()
bri_obj = BRISQUE(url=False)

def extract_base_features(mask):
    """Compute geometric, Zernike, Fourier, and texture shape descriptors from a binary mask."""
    
    features = ["area", "perimeter", "aspect_ratio", "extent", "solidity", "eccentricity", 
        "orientation", "circularity", "elongation", "compactness"]
    
    if mask is None or mask.sum() == 0:
        return {f: 0 for f in features}

    # --- Region properties ---
    # mask = mask.astype(np.uint8)
    labeled = label(mask)
    props = regionprops(labeled)
    if len(props) == 0:
        return {f: 0 for f in features}
    p = props[0]
    major_axis = p.major_axis_length
    minor_axis = p.minor_axis_length

    # ----- base shape features -----
    area = p.area
    perimeter = max(p.perimeter, 1e-6) # Ignoring too small perimeters
    aspect_ratio = major_axis / minor_axis if minor_axis > 0 else 0 # L_major / L_minor
    extent = p.extent
    solidity = p.solidity
    eccentricity = p.eccentricity
    orientation = p.orientation
    circularity = 4 * np.pi * area / (perimeter ** 2)
    elongation = 1 - (minor_axis / major_axis) if major_axis > 0 else 0
    # convexity = p.perimeter_convex / perimeter
    compactness = (perimeter ** 2) / (4 * np.pi * area + 1e-6)

    # ----- Assemble features -----
    features_d = {
        "area": area,
        "perimeter": perimeter,
        "aspect_ratio": aspect_ratio,
        "extent": extent,
        "solidity": solidity,
        "eccentricity": eccentricity,
        "orientation": orientation,
        "circularity": circularity,
        "elongation": elongation,
        "compactness": compactness
    }
    return features_d

def compute_sift_features(image, mask=None):
    # if isinstance(image, torch.Tensor):
    #     image = image.detach().cpu().numpy().transpose(1, 2, 0) # Transform tensor to numpy image
    # if isinstance(mask, torch.Tensor):
    #     mask = mask.numpy().astype(np.uint8)
    gray= cv2.cvtColor(img_as_ubyte(image), cv2.COLOR_RGB2GRAY) # converts image into uint8 and mask as input
    keypoints, descriptors = sift.detectAndCompute(gray, mask)
    if descriptors is None:
        descriptors = np.full((0, 128), np.nan, dtype=np.float32)  # return empty array if no keypoints
    return keypoints, descriptors

def compute_orb_features(image, mask=None):
    # if isinstance(image, torch.Tensor):
    #     image = image.detach().cpu().numpy().transpose(1, 2, 0) # Transform tensor to numpy image
    # if isinstance(mask, torch.Tensor):
    #     mask = mask.numpy().astype(np.uint8)
    gray= cv2.cvtColor(img_as_ubyte(image), cv2.COLOR_RGB2GRAY)
    keypoints, descriptors = orb.detectAndCompute(gray, mask)
    if descriptors is None:
        descriptors = np.full((0, 32), np.nan, dtype=np.float32)  # return empty array if no keypoints
    return keypoints, descriptors

def compute_hu_moments(mask):
    # if not isinstance(mask, np.ndarray):
    #     mask = mask.numpy().astype(np.uint8)
    moments = cv2.moments(mask)
    hu = cv2.HuMoments(moments).flatten()
    hu = np.log(np.abs(hu) + 1e-12) # log-scale for stability
    return hu

def compute_zernike_moments(mask, degree=8):
    # if not isinstance(mask, np.ndarray):
    #     mask = mask.numpy().astype(np.uint8)
    radius = min(mask.shape) // 2
    mask_norm = mask / mask.max() if mask.max() > 0 else mask
    zern = zernike_moments(mask_norm, radius=radius, degree=degree)
    return zern

# *** Updated fourier descriptors (Dec 4, 2025)
def compute_fourier_descriptors(mask, image=None, fourier_harmonics=20, visualize=False):
    if not isinstance(mask, np.ndarray): # Ensure proper mask format
        mask = mask.numpy().astype(np.uint8)
    # --- 2. Find largest contour (object part) ---
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not contours:
        return np.full(fourier_harmonics, np.nan, dtype=np.float32)
    cnt = max(contours, key=cv2.contourArea)
    if len(cnt) < 3:
        # Too few points for Fourier transform
        return np.full(fourier_harmonics, np.nan, dtype=np.float32)
    
    # Translation invariance: center contour
    complex_contour = cnt[:,0,0] + 1j * cnt[:,0,1]
    fd = np.fft.fft(complex_contour)
    
    if visualize: # ** IMPORTANT: Visualization uses raw contour (so you can see the real shape), descriptors are centered.
        # Convert image if needed
        H, W = mask.shape
        if image is not None:
            if isinstance(image, torch.Tensor):
                image = image.detach().cpu().numpy().transpose(1, 2, 0)
            elif isinstance(image, Image.Image):
                image = np.array(image.convert('RGB'))
            elif image.dtype != np.uint8:  # NumPy float → uint8
                image = (image*255).astype(np.uint8)
            img_draw = image.copy()
        else:
            img_draw = np.zeros((H, W, 3), dtype=np.uint8)
        cv2.drawContours(img_draw, [cnt.astype(np.int32)], -1, (0, 255, 0), 2)

        fd_recon = fd.copy()
        keep = fourier_harmonics
        if 2 * keep < len(fd_recon):
            fd_recon[keep:-keep] = 0 # Safe zeroing
        else:
            fd_recon[keep:] = 0
        recon = np.fft.ifft(fd_recon)
        pts = np.column_stack((recon.real, recon.imag)).astype(np.int32)

        for i in range(len(pts)):
            cv2.line(img_draw, tuple(pts[i]), tuple(pts[(i + 1) % len(pts)]), (255, 0, 255), 1)
        plt.figure(figsize=(16, 6))
        plt.imshow(img_draw)
        plt.axis('off')
        plt.title("Shape Descriptors Overlay")
        plt.legend(
            handles=[
                Patch(facecolor='green', edgecolor='green'),
                Patch(facecolor='magenta', edgecolor='magenta')
            ],
            labels=["Contour", "Fourier Reconstruction"],
            loc='upper right'
        )
        plt.show()
    
    cnt_centered = complex_contour - np.mean(complex_contour)
    fd = np.fft.fft(cnt_centered)
    if len(fd) < 2 or np.abs(fd[1]) == 0:
        return np.full(fourier_harmonics, np.nan, dtype=np.float32)

    # Scale invariance: divide by first descriptor magnitude
    fd = fd / np.abs(fd[1])

    # Rotation invariance: use only magnitudes
    fd_normalized = np.abs(fd)

    # Keep only first N harmonics
    fd_truncated = fd_normalized[:fourier_harmonics]
    if len(fd_truncated) < fourier_harmonics:
        fd_truncated = np.concatenate([fd_truncated, np.full((fourier_harmonics - len(fd_truncated)), np.nan)])
    return fd_truncated

def extract_shape_features(image, mask):
    # Compute base features
    features = extract_base_features(mask)

    # Compute sift features
    sift_kp, sift_ds = compute_sift_features(image, mask)
    sift_sizes = [k.size for k in sift_kp]
    if sift_ds.shape[0] > 0:
        sift_mean_ds = np.nanmean(sift_ds, axis=0)
    else:
        sift_mean_ds = np.full(128, np.nan)
    sift_dict = {f'sift_ds{i+1}': sift_mean_ds[i] for i in range(len(sift_mean_ds))}
    sift_dict['sift_kp_n'] = len(sift_kp)
    sift_dict['sift_kp_size'] = np.max(sift_sizes) if sift_sizes else 0

    # Compute orb features
    orb_kp, orb_ds = compute_orb_features(image, mask)
    if orb_ds.shape[0] > 0:
        orb_mean_ds = np.nanmean(orb_ds, axis=0)
    else:
        orb_mean_ds = np.full(32, np.nan)
    orb_dict = {f'orb_ds{i+1}': orb_mean_ds[i] for i in range(len(orb_mean_ds))}
    orb_dict['orb_kp_n'] = len(orb_kp)

    # Compute hu moments
    hu_moments = compute_hu_moments(mask)
    hu_dict = {f"hu{i+1}": hu_moments[i] for i in range(len(hu_moments))}

    # Compute Zernike moments
    zern_moments = compute_zernike_moments(mask, degree=8)
    zern_dict = {f"zernike_{i+1}": zern_moments[i] for i in range(len(zern_moments))}

    # Compute fourier descriptors
    fourier_ds = compute_fourier_descriptors(mask, fourier_harmonics=20)
    fourier_dict = {f"fourier_{i+1}": fourier_ds[i] for i in range(len(fourier_ds))}

    features.update(sift_dict)
    features.update(orb_dict)
    features.update(hu_dict)
    features.update(zern_dict)
    features.update(fourier_dict)
    converted = {k: np.float32(v) for k, v in features.items()}
    return converted

def extract_visual_features(image, mask):
    # --- 1. Ensure binary uint8 mask ---
    if not isinstance(mask, np.ndarray):
        mask = mask.numpy().astype(np.uint8)
    # Convert image to numpy
    if isinstance(image, torch.Tensor):
        image = image.detach().cpu().numpy().transpose(1, 2, 0)
    elif isinstance(image, Image.Image):
        image = np.array(image.convert('RGB'))
    img_cropped = np.zeros_like(image)
    img_cropped[mask==1] = image[mask==1]
    # plt.imshow(img_cropped)

    # --- Brightness ---
    brightness = np.mean(img_cropped)

    # --- Contrast (standard deviation of luminance) ---
    gray = rgb2gray(img_cropped)
    contrast = np.std(gray)

    # --- Sharpness (variance of Laplacian) ---
    gray_8u = (gray * 255).astype(np.uint8)
    lap_var = cv2.Laplacian(gray_8u, cv2.CV_64F).var()

    # --- Colorfulness (Hasler & Süsstrunk, 2003) ---
    (R, G, B) = cv2.split(img_cropped)
    rg = np.abs(R - G)
    yb = np.abs(0.5 * (R + G) - B)
    std_rg, std_yb = np.std(rg), np.std(yb)
    mean_rg, mean_yb = np.mean(rg), np.mean(yb)
    colorfulness = np.sqrt(std_rg**2 + std_yb**2) + 0.3 * np.sqrt(mean_rg**2 + mean_yb**2)

    # --- Entropy (texture complexity) ---
    entropy = shannon_entropy(gray)

    # BRISQUE
    bri_obj = BRISQUE(url=False)
    brisque_score = bri_obj.score(img_cropped)

    # NIQE
    niqe_score = niqe(gray)

    # PIQE
    piqe_score, activityMask, noticeableArtifactMask, noiseMask = piqe(gray)

    # --- Aggregate descriptors ---
    descriptors = {
        "brightness": np.float32(brightness),
        "contrast": np.float32(contrast),
        "sharpness": np.float32(lap_var),
        "colorfulness": np.float32(colorfulness),
        "entropy": np.float32(entropy),
        "brisque": np.float32(brisque_score),
        "niqe": np.float32(niqe_score.item()),
        "piqe": np.float32(piqe_score)
    }
    return descriptors

def extract_combined_features(image, mask): 
    # ---- Convert once ----
    if isinstance(image, torch.Tensor):
        image = image.detach().cpu().numpy().transpose(1, 2, 0)
    if isinstance(mask, torch.Tensor):
        mask = mask.detach().cpu().numpy().astype(np.uint8)
    mask_u8 = mask.astype(np.uint8)
    image_f = (image - image.min()) / (image.max() - image.min() + 1e-8) # Linearly rescale to [0, 1] and avoid division by zero
    image_u8 = img_as_ubyte(image_f)

    combined_features = extract_shape_features(image_u8, mask_u8)
    vis_features = extract_visual_features(image_f, mask_u8)
    combined_features.update(vis_features)
    return combined_features

def extract_all_features(image, mask):
    abdomen_mask = mask == 1
    head_mask = mask == 2
    thorax_mask = mask == 3
    full_mask = mask > 0

    head_feats = extract_combined_features(image, head_mask)
    thorax_feats = extract_combined_features(image, thorax_mask)
    abdomen_feats = extract_combined_features(image, abdomen_mask)
    body_feats = extract_combined_features(image, full_mask)

    # Ratios
    area_sum = head_feats["area"] + thorax_feats["area"] + abdomen_feats["area"]
    ratios = {
        "head_to_thorax_area": head_feats["area"] / (thorax_feats["area"] + 1e-6),
        "thorax_to_abdomen_area": thorax_feats["area"] / (abdomen_feats["area"] + 1e-6),
        "head_to_total_area": head_feats["area"] / (area_sum + 1e-6),
        "thorax_to_total_area": thorax_feats["area"] / (area_sum + 1e-6),
        "abdomen_to_total_area": abdomen_feats["area"] / (area_sum + 1e-6),
    }

    record = {}
    record.update({f"head_{k}": v for k, v in head_feats.items()})
    record.update({f"thorax_{k}": v for k, v in thorax_feats.items()})
    record.update({f"abdomen_{k}": v for k, v in abdomen_feats.items()})
    record.update({f"full_{k}": v for k, v in body_feats.items()})
    record.update(ratios)
    return record

As discussed, each part mask has 225 shape descriptors and 8 visual descriptors. We have five values for inter-part ratios: head_to_thorax, thorax_to_abdomen, head_to_total, thorax_to_total and abdomen_to_total. Therefore, total number of descriptors for all parts (head, thorax, abdomen, fullbody) = (225+8)*4 + 5 = 932 + 5 = 937

In [4]:
class PartWholeDataset(Dataset):
    def __init__(self, root, images_dir="images", masks_dir="masks", image_size=320):
        self.images_dir = os.path.join(root, images_dir)
        self.masks_dir = os.path.join(root, masks_dir)
        self.image_paths = sorted([os.path.join(self.images_dir, f) for f in os.listdir(self.images_dir)])
        self.image_size = image_size

        # transform for image
        self.img_transform = transforms.Compose([
            transforms.Resize((image_size, image_size), interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.ToTensor(),
            # transforms.Normalize(mean=[0.5016, 0.4647, 0.3782], std=[0.2738, 0.2654, 0.2885]) # Computed using the find_mean_and_sd_of_partwhole_dataset script
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        base = os.path.splitext(os.path.basename(img_path))[0]
        mask_path = os.path.join(self.masks_dir, base + "_m.png")

        img = Image.open(img_path).convert("RGB")
        img = self.img_transform(img)
        mask = Image.open(mask_path).convert("L")
        mask = mask.resize((self.image_size, self.image_size), resample=Image.NEAREST)
        mask = torch.from_numpy(np.array(mask, dtype=np.int64))
        return img, mask, img_path

In [5]:
# Load the datasets
DATA_DIR = r"/home/c/choton/beemachine/datasets/Beemachine_Partwhole_v5/"
train_path = os.path.join(DATA_DIR, "train") # Path for the training data
val_path = os.path.join(DATA_DIR, "valid") # Path for validation data
test_path = os.path.join(DATA_DIR, "test") # Path for testing data

train_dataset = PartWholeDataset(root=train_path, images_dir="aug_images", masks_dir="aug_masks", image_size=IMAGE_SIZE)
val_dataset = PartWholeDataset(root=val_path, image_size=IMAGE_SIZE)
test_dataset = PartWholeDataset(root=test_path, image_size=IMAGE_SIZE)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

print(f"Images in dataset, train: {len(train_dataset)}, val: {len(val_dataset)}, test: {len(test_dataset)}")

Images in dataset, train: 34722, val: 1158, test: 771


In [6]:
def extract_features_for_dataset(dataset, output_csv):
    records = []
    print(f"Extracting shape features for {len(dataset)} images...")
    for image, mask, img_path in tqdm(dataset):
        features = extract_all_features(image, mask)
        record = {"image": os.path.basename(img_path)}
        record.update(features)
        records.append(record)

    df = pd.DataFrame(records)
    df.to_csv(output_csv, index=False)
    print(f"✅ Saved shape features to: {output_csv}")
    return df

In [7]:
feat_path = r"./bee_gt_shape_features_concise"
os.makedirs(feat_path, exist_ok=True)

In [8]:
train_csv = os.path.join(feat_path, r"bee_gt_features_train.csv")
df = extract_features_for_dataset(train_dataset, train_csv)
df

Extracting shape features for 34722 images...


  0%|          | 6/34722 [00:06<9:21:10,  1.03it/s] /home/c/choton/miniconda3/envs/beemachine_new/lib/python3.11/site-packages/brisque/brisque.py:114: RuntimeWarning: invalid value encountered in scalar divide
  return (np.sum(np.abs(x)) / size) ** 2 / (np.sum(x ** 2) / size)
/home/c/choton/miniconda3/envs/beemachine_new/lib/python3.11/site-packages/brisque/brisque.py:124: RuntimeWarning: invalid value encountered in divide
  return squares_sum / ((filtered_values.shape))
/home/c/choton/miniconda3/envs/beemachine_new/lib/python3.11/site-packages/pypiqe/piqe.py:142: RuntimeWarning: invalid value encountered in divide
  ipImage = np.round(255 * (ipImage / np.max(ipImage)))
100%|██████████| 34722/34722 [12:38:34<00:00,  1.31s/it]  


✅ Saved shape features to: ./bee_gt_shape_features_concise/bee_gt_features_train.csv


,image,head_area,head_perimeter,head_aspect_ratio,head_extent,head_solidity,head_eccentricity,head_orientation,head_circularity,head_elongation,...,full_colorfulness,full_entropy,full_brisque,full_niqe,full_piqe,head_to_thorax_area,thorax_to_abdomen_area,head_to_total_area,thorax_to_total_area,abdomen_to_total_area
0,0090L0W0TQIQUR0Q3RIQCRIQTRQQS00QL0P0OQLQJRSQ9R...,4330.0,608.617310,3.586139,0.248822,0.568018,0.960334,1.295621,0.146896,0.721149,...,0.129306,6.401090,60.094749,27.640350,41.000252,0.143003,23.202299,0.120566,0.843097,0.036337
1,0090L0W0TQIQUR0Q3RIQCRIQTRQQS00QL0P0OQLQJRSQ9R...,4325.0,606.824402,3.585948,0.248535,0.568854,0.960330,1.296722,0.147594,0.721134,...,0.129675,6.392323,60.289352,24.562759,43.832634,0.142942,14.504794,0.117950,0.825161,0.056889
2,0090L0W0TQIQUR0Q3RIQCRIQTRQQS00QL0P0OQLQJRSQ9R...,4327.0,606.824402,3.584670,0.248650,0.569117,0.960301,-1.296543,0.147663,0.721034,...,0.129652,6.391943,47.546585,27.722122,44.148281,0.143041,23.056402,0.120566,0.842877,0.036557
3,0090L0W0TQIQUR0Q3RIQCRIQTRQQS00QL0P0OQLQJRSQ9R...,4328.0,608.617310,3.587408,0.248707,0.567755,0.960363,-1.295801,0.146828,0.721247,...,0.129309,6.400627,47.292450,24.678831,41.000793,0.142928,14.551177,0.117961,0.825320,0.056718
4,0090L0W0TQIQUR0Q3RIQCRIQTRQQS00QL0P0OQLQJRSQ9R...,4332.0,608.617310,3.587018,0.248937,0.568206,0.960354,0.274950,0.146964,0.721217,...,0.129716,6.398315,56.920498,27.667301,43.878483,0.143164,23.063263,0.120658,0.842799,0.036543
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34717,Bombus_zonatus_GBIF_iNat_3058832608_2-jpg_jpg....,4248.0,488.439636,1.840455,0.564819,0.622965,0.839510,1.436783,0.223755,0.456656,...,0.167198,9.348055,35.663631,25.897465,37.843678,0.106854,3.173799,0.075147,0.703267,0.221585
34718,Bombus_zonatus_GBIF_iNat_3058832608_2-jpg_jpg....,4248.0,488.439636,1.840455,0.564819,0.622965,0.839510,-1.436783,0.223755,0.456656,...,0.167152,9.316640,53.308208,26.743814,37.511600,0.106854,3.173799,0.075147,0.703267,0.221585
34719,Bombus_zonatus_GBIF_iNat_3058832608_2-jpg_jpg....,4242.0,489.025421,1.842837,0.564021,0.622268,0.839964,-1.435658,0.222904,0.457359,...,0.167130,9.353924,52.911057,25.226418,37.833157,0.106682,3.176212,0.075048,0.703471,0.221481
34720,Bombus_zonatus_GBIF_iNat_3058832608_2-jpg_jpg....,4245.0,488.439636,1.842244,0.564420,0.622708,0.839851,0.134523,0.223597,0.457184,...,0.167118,9.333622,37.994225,25.915386,35.862720,0.106760,3.175371,0.075094,0.703391,0.221515


In [9]:
val_csv = os.path.join(feat_path, r"bee_gt_features_val.csv")
df = extract_features_for_dataset(val_dataset, val_csv)
df

Extracting shape features for 1158 images...


  2%|▏         | 23/1158 [00:21<17:21,  1.09it/s]/home/c/choton/miniconda3/envs/beemachine_new/lib/python3.11/site-packages/brisque/brisque.py:114: RuntimeWarning: invalid value encountered in scalar divide
  return (np.sum(np.abs(x)) / size) ** 2 / (np.sum(x ** 2) / size)
/home/c/choton/miniconda3/envs/beemachine_new/lib/python3.11/site-packages/brisque/brisque.py:124: RuntimeWarning: invalid value encountered in divide
  return squares_sum / ((filtered_values.shape))
/home/c/choton/miniconda3/envs/beemachine_new/lib/python3.11/site-packages/pypiqe/piqe.py:142: RuntimeWarning: invalid value encountered in divide
  ipImage = np.round(255 * (ipImage / np.max(ipImage)))
100%|██████████| 1158/1158 [17:57<00:00,  1.07it/s]


✅ Saved shape features to: ./bee_gt_shape_features_concise/bee_gt_features_val.csv


,image,head_area,head_perimeter,head_aspect_ratio,head_extent,head_solidity,head_eccentricity,head_orientation,head_circularity,head_elongation,...,full_colorfulness,full_entropy,full_brisque,full_niqe,full_piqe,head_to_thorax_area,thorax_to_abdomen_area,head_to_total_area,thorax_to_total_area,abdomen_to_total_area
0,00903Q80CQ70TQX0AR403QU0Q050BRZQTQJKWR70JQN0YQ...,5410.0,434.575684,1.484547,0.497243,0.739172,0.739091,-1.247538,0.359978,0.326394,...,0.154188,6.574201,69.938728,28.917168,43.730812,0.180586,16.606430,0.145540,0.805929,0.048531
1,0090L09000N0Q0SQQ080FQFKCQHQ1R3K1RIQH0N0L0SQOR...,2058.0,449.345245,1.546949,0.172564,0.321864,0.762970,0.208325,0.128084,0.353566,...,0.120873,4.861553,66.378494,25.980085,47.686066,0.129605,1.430412,0.070873,0.546835,0.382292
2,0H6RLHERTZIZ2LRZWLZZ6LYLELYL2LHZFZ0R2L0RRHSRRH...,3315.0,355.989899,1.162912,0.458697,0.689332,0.510445,-1.497345,0.328714,0.140089,...,0.134285,7.166333,40.585583,25.544678,38.450874,0.109363,3.333187,0.077597,0.709534,0.212870
3,0HGRDZIROZ7RFZXRULHZ9L0R3ZYLNLKROZRZ9LYLNLZZTZ...,608.0,173.633514,8.116632,0.405063,0.683914,0.992381,1.451794,0.253423,0.876796,...,0.063711,6.655105,32.683594,25.874298,41.878220,0.018505,72.210991,0.017925,0.968661,0.013414
4,0K9KLKWKIKT0UQLSPQA0UQB0GQB04Q6K2Q2K6QUKPQC0KK...,8.0,8.000000,2.236068,1.000000,1.000000,0.894427,1.570796,1.570796,0.552786,...,0.125911,5.211917,60.000885,23.613640,57.017181,0.000322,10.280810,0.000293,0.911087,0.088620
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1153,Bombus_sylvestris_iNat_21848423_1-jpg_jpg.rf.0...,3480.0,467.937134,1.185989,0.272492,0.429100,0.537634,-0.206154,0.199717,0.156822,...,0.177342,7.189582,62.782967,24.507561,58.299995,0.142676,1.522059,0.079278,0.555654,0.365067
1154,Bombus_sylvestris_iNat_27771888_01-jpg_jpg.rf....,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.102771,5.752216,89.865562,26.728430,69.970680,0.000000,0.890869,0.000000,0.471143,0.528857
1155,Bombus_sylvestris_iNat_36837121_01-jpg_jpg.rf....,4070.0,504.611206,1.798342,0.339167,0.596337,0.831137,0.148295,0.200859,0.443932,...,0.171758,7.327914,60.891003,27.185129,51.417065,2.110996,0.075445,0.128989,0.061104,0.809907
1156,Bombus_sylvestris_iNat_47607375_1-jpg_jpg.rf.3...,5225.0,453.345245,1.978194,0.424797,0.698810,0.862820,-1.515664,0.319476,0.494488,...,0.090634,7.893108,30.526089,28.071922,33.448742,1.948173,0.108028,0.159620,0.081933,0.758447


In [10]:
test_csv = os.path.join(feat_path, r"bee_gt_features_test.csv")
df = extract_features_for_dataset(test_dataset, test_csv)
df

Extracting shape features for 771 images...


  1%|          | 4/771 [00:03<12:07,  1.05it/s]/home/c/choton/miniconda3/envs/beemachine_new/lib/python3.11/site-packages/brisque/brisque.py:114: RuntimeWarning: invalid value encountered in scalar divide
  return (np.sum(np.abs(x)) / size) ** 2 / (np.sum(x ** 2) / size)
/home/c/choton/miniconda3/envs/beemachine_new/lib/python3.11/site-packages/brisque/brisque.py:124: RuntimeWarning: invalid value encountered in divide
  return squares_sum / ((filtered_values.shape))
/home/c/choton/miniconda3/envs/beemachine_new/lib/python3.11/site-packages/pypiqe/piqe.py:142: RuntimeWarning: invalid value encountered in divide
  ipImage = np.round(255 * (ipImage / np.max(ipImage)))
100%|██████████| 771/771 [12:46<00:00,  1.01it/s]


✅ Saved shape features to: ./bee_gt_shape_features_concise/bee_gt_features_test.csv


,image,head_area,head_perimeter,head_aspect_ratio,head_extent,head_solidity,head_eccentricity,head_orientation,head_circularity,head_elongation,...,full_colorfulness,full_entropy,full_brisque,full_niqe,full_piqe,head_to_thorax_area,thorax_to_abdomen_area,head_to_total_area,thorax_to_total_area,abdomen_to_total_area
0,0090L0W0TQIQURLQ9RSQ3RKQTRQQOR7Q3R60CQG0OQ80FQ...,4279.0,506.326935,1.777138,0.443144,0.666926,0.826660,-1.310267,0.209744,0.437298,...,0.076993,6.121056,63.953552,29.595713,47.932377,0.198838,1.826825e+00,0.113867,0.572660,0.313473
1,0090Q0W0H0X0YQYKWR40S0SQURI0JQ70ARZQAR70CQHQUR...,2892.0,221.480225,1.697738,0.708824,0.953511,0.808119,-0.267619,0.740864,0.410981,...,0.135423,7.187157,72.190109,29.266737,54.327305,0.088424,8.546120e+00,0.073354,0.829575,0.097070
2,0K6KRKPKIKOK4KVKVQO08QPKKKD05QD0EQ9KSKA04QC04Q...,9156.0,400.450806,1.215336,0.726667,0.960151,0.568306,0.103841,0.717492,0.177182,...,0.136263,5.984114,69.195908,26.606197,63.917778,0.377038,2.611183e+02,0.273044,0.724182,0.002773
3,0K9KLKWKIKT0UQB0PQLSWQF0NQY05KAKVQY05QC05Q6KQK...,2169.0,453.037628,1.134231,0.156178,0.303145,0.471896,1.255855,0.132801,0.118346,...,0.132100,5.446256,79.839951,25.544371,59.388435,0.096606,1.111485e+01,0.081416,0.842761,0.075823
4,0L6ZLLEZILTH2HAHIHDHUHYHNHZREHJH7LNZMLFHWHVZQL...,7078.0,669.788879,1.372911,0.538005,0.701695,0.685174,-0.135562,0.198264,0.271621,...,0.220103,6.088460,67.306000,27.478662,61.603825,0.247439,2.860500e+10,0.198358,0.801642,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
766,Bombus_trifasciatus_iNat_95727364_01-jpg_jpg.r...,7539.0,592.037598,2.363889,0.505159,0.795589,0.906115,-1.441886,0.270287,0.576968,...,0.162338,7.466583,53.445713,25.307562,48.816837,20.542234,1.919255e-02,0.278933,0.013579,0.707489
767,Bombus_zonatus_GBIF_iNat_2236827425_2-jpg_jpg....,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,NaN,27.075386,100.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000
768,Bombus_zonatus_GBIF_iNat_2236827425_3-jpg_jpg....,4149.0,622.222412,1.287401,0.235124,0.330361,0.629799,1.463304,0.134667,0.223242,...,0.185269,4.817892,63.231586,26.097486,42.069763,0.334867,1.071614e+00,0.147646,0.440910,0.411444
769,Bombus_zonatus_GBIF_iNat_3058832608_1-jpg_jpg....,4461.0,550.499573,1.795978,0.319579,0.517457,0.830647,0.541462,0.184981,0.443200,...,0.157267,7.733049,58.705132,27.434587,42.797489,1.946335,1.165463e-01,0.168856,0.086756,0.744389
